# Lesson 02 - Explorar o Microsoft Agent Framework

O **Microsoft Agent Framework (MAF)** é uma estrutura unificada para construir agentes de IA. Fornece uma arquitetura limpa e componível com quatro blocos de construção principais:

- **Cliente** – liga-se a um endpoint de modelo de IA e gere a comunicação
- **Agente** – envolve um cliente com instruções e definições de ferramentas
- **Ferramentas** – alargam as capacidades do agente com funções personalizadas que o modelo pode chamar
- **Sessão** – mantém o histórico da conversa para interações multi-turno

Nesta lição, vamos construir um **agente de reserva de viagens** que verifica a disponibilidade do destino usando estes conceitos.


## Configuração


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Compreendendo a Arquitectura do Framework do Agente

O Microsoft Agent Framework segue uma arquitectura em camadas:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Cliente** – Um `AzureAIProjectAgentProvider` liga-se a uma implementação Azure OpenAI. Trata da autenticação, formatação do pedido, e análise da resposta.
2. **Agente** – Criado a partir do cliente através de `provider.create_agent()`, o agente combina o acesso ao modelo com instruções (prompt do sistema) e ferramentas.
3. **Ferramentas** – Funções Python decoradas com `@tool` que o agente pode invocar para realizar acções ou obter dados.
4. **Sessão** – Um objecto `AgentSession` (criado através de `agent.create_session()`) que guarda o histórico da conversa, permitindo diálogos de múltiplas voltas onde o agente se lembra do contexto anterior.

Vamos construir cada camada passo a passo.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Adicionar Ferramentas com o Decorador @tool

As ferramentas permitem que os agentes realizem ações para além de gerar texto. O decorador `@tool` converte uma função Python comum numa coisa que o agente pode chamar.

Pontos-chave:
- Use `Annotated[type, "description"]` para que o modelo compreenda cada parâmetro.
- A docstring torna-se na descrição da ferramenta que o modelo vê.
- `approval_mode="never_require"` significa que a ferramenta é executada automaticamente sem confirmação do utilizador.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Criar um Agente com Ferramentas

Agora combinamos o cliente, as instruções e as ferramentas num agente. As `instructions` funcionam como prompt do sistema — definem a persona e o comportamento do agente.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Conversas Multi-Turno com Sessões

Uma `AgentSession` (criada através de `agent.create_session()`) guarda todas as mensagens numa conversa. Ao passar a mesma sessão a cada chamada de `agent.run()`, o agente tem acesso ao histórico completo da conversa e pode referir-se a mensagens anteriores.

Passamos `tools=[check_destination_availability]` para que o agente possa usar o nosso verificador de disponibilidade durante cada turno.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Resumo

Nesta lição explorou os quatro pilares do Microsoft Agent Framework:

| Conceito | O Que Aprendeu |
|---------|------------------|
| **Cliente** | `AzureAIProjectAgentProvider` liga-se ao Azure OpenAI com autenticação baseada em credenciais |
| **Agente** | `provider.create_agent()` agrupa uma ligação a um modelo com instruções e um nome |
| **Ferramentas** | O decorador `@tool` expõe funções Python para o agente chamar |
| **Sessão** | `agent.create_session()` mantém o histórico da conversa ao longo de múltiplas interações |

Estes blocos de construção compõem-se juntos para criar agentes que podem manter conversas naturais, chamar funções externas e manter contexto — a base para padrões agentic mais avançados em lições futuras.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos para assegurar a precisão, tenha em atenção que traduções automáticas podem conter erros ou imprecisões. O documento original no seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, é recomendada a tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
